<a href="https://colab.research.google.com/github/sanjjj/ml-specialization-journey/blob/main/Apartment_locating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
#  AUSTIN APARTMENT LEAD SCORER
#  Complete Classification Model — Everything We Learned
#
#  Concepts used:
#  ✓ Scalars, vectors, matrix
#  ✓ Indexing and slicing
#  ✓ Feature scaling (z-score)
#  ✓ Sigmoid function
#  ✓ Prediction f(x) = sigmoid(np.dot(X_n, w) + b)
#  ✓ Cross-entropy cost function
#  ✓ Gradients dJ/dw and dJ/db
#  ✓ Gradient descent loop
#  ✓ Decision boundary
#  ✓ Predict new leads
# ============================================================

import numpy as np

print("=" * 55)
print("  AUSTIN APARTMENT LEAD SCORER")
print("  Built from scratch using what we learned")
print("=" * 55)


# ── STEP 1: TRAINING DATA ────────────────────────────────────
# 8 past leads you already know the outcome of.
# Features: [budget ($k), urgency (1-5), reply time (hrs)]
# y = 1 means they signed. y = 0 means they ghosted.

print("\n📋 STEP 1: Training Data")
print("-" * 40)

X = np.array([          # MATRIX — shape (8, 3)
    [2.5, 5, 0.5],      # Maria  → signed
    [1.2, 1, 48 ],      # Jake   → ghosted
    [2.8, 4, 2  ],      # Sarah  → signed
    [1.5, 2, 24 ],      # Mike   → ghosted
    [3.0, 5, 1  ],      # Lisa   → signed
    [1.1, 1, 36 ],      # Tom    → ghosted
    [2.3, 4, 3  ],      # Amy    → signed
    [1.4, 2, 20 ],      # Dave   → ghosted
])

y = np.array([1, 0, 1, 0, 1, 0, 1, 0])  # VECTOR — 1=signed, 0=ghosted

m, n = X.shape   # m=8 leads (SCALAR), n=3 features (SCALAR)
names = ['Maria','Jake','Sarah','Mike','Lisa','Tom','Amy','Dave']

print(f"X shape: {X.shape}  ← {m} leads, {n} features each")
print(f"y shape: {y.shape}  ← {m} outcomes")
print(f"\nFirst lead (indexing):  X[0] = {X[0]}  ← Maria")
print(f"Last lead  (indexing):  X[-1] = {X[-1]}  ← Dave")
print(f"All budgets (slicing):  X[:,0] = {X[:,0]}")
print(f"Signed leads (slicing): X[:1] = {X[:1]}")


# ── STEP 2: FEATURE SCALING ──────────────────────────────────
# Budget (1.1–3.0), urgency (1–5), reply (0.5–48) all different
# scales. Z-score normalizes them all to roughly -2 to +2.
# Formula: x_scaled = (x - mean) / std_dev

print("\n\n📐 STEP 2: Feature Scaling (Z-score)")
print("-" * 40)

mu    = X.mean(axis=0)   # VECTOR — mean of each column
sigma = X.std(axis=0)    # VECTOR — std dev of each column
X_n   = (X - mu) / sigma # MATRIX — scaled version of X

print(f"mu    = {mu.round(3)}")
print(f"         budget={mu[0]:.3f}  urgency={mu[1]:.3f}  reply={mu[2]:.3f}")
print(f"\nsigma = {sigma.round(3)}")
print(f"         budget={sigma[0]:.3f}  urgency={sigma[1]:.3f}  reply={sigma[2]:.3f}")

print(f"\nHow we got mu[0] = {mu[0]:.3f}:")
total = sum([2.5,1.2,2.8,1.5,3.0,1.1,2.3,1.4])
print(f"  2.5+1.2+2.8+1.5+3.0+1.1+2.3+1.4 = {total}")
print(f"  {total} / 8 = {total/8:.3f}")

print(f"\nMaria raw:    {X[0]}")
print(f"Maria scaled: {X_n[0].round(3)}")
print(f"  budget:  ({X[0][0]} - {mu[0]:.3f}) / {sigma[0]:.3f} = {X_n[0][0]:.3f}")
print(f"  urgency: ({X[0][1]} - {mu[1]:.3f}) / {sigma[1]:.3f} = {X_n[0][1]:.3f}")
print(f"  reply:   ({X[0][2]} - {mu[2]:.3f}) / {sigma[2]:.3f} = {X_n[0][2]:.3f}")

print(f"\nFull scaled matrix X_n:")
for i, name in enumerate(names):
    outcome = "signed ✓" if y[i]==1 else "ghosted ✗"
    print(f"  {name:<6}: {X_n[i].round(2)}  ← {outcome}")


# ── STEP 3: SIGMOID FUNCTION ─────────────────────────────────
# Squishes any number into 0-1 range.
# Output = probability of signing.
# sigmoid(big negative) → near 0  (won't sign)
# sigmoid(0)            → 0.5     (50/50)
# sigmoid(big positive) → near 1  (will sign)

print("\n\n📈 STEP 3: Sigmoid Function")
print("-" * 40)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("sigmoid(z) = 1 / (1 + e^-z)")
print(f"\nTest values:")
for z in [-4, -2, -1, 0, 1, 2, 4]:
    sig = sigmoid(z)
    bar = "▓" * int(sig * 20)
    print(f"  sigmoid({z:+d}) = {sig:.3f}  {bar}")


# ── STEP 4: PREDICT ──────────────────────────────────────────
# f(x) = sigmoid(np.dot(X_n, w) + b)
# Same np.dot as your Austin house model.
# Just wrapped in sigmoid to get a probability.

print("\n\n🔮 STEP 4: The Model — predict(X, w, b)")
print("-" * 40)

def predict(X, w, b):
    z = np.dot(X, w) + b   # linear step — same as regression
    return sigmoid(z)       # squish to 0-1

# Before training: w=[0,0,0], b=0
w = np.zeros(n)   # VECTOR — [0, 0, 0]
b = 0.0           # SCALAR

preds_before = predict(X_n, w, b)
print(f"w = {w}  b = {b}  (before training)")
print(f"\nPredictions before training:")
for i, name in enumerate(names):
    print(f"  {name:<6}: {preds_before[i]:.3f} = 50%  (model has no idea yet)")

print(f"\nHow np.dot works for Maria:")
print(f"  z = {w[0]}×{X_n[0][0]:.2f} + {w[1]}×{X_n[0][1]:.2f} + {w[2]}×{X_n[0][2]:.2f} + {b}")
print(f"  z = 0  →  sigmoid(0) = 0.500")


# ── STEP 5: COST FUNCTION ────────────────────────────────────
# Cross-entropy — measures how wrong the probabilities are.
# J = -mean( y*log(f) + (1-y)*log(1-f) )
# Lower J = better model.

print("\n\n📉 STEP 5: Cost Function J (Cross-Entropy)")
print("-" * 40)

def compute_cost(X, y, w, b):
    m    = X.shape[0]
    f    = predict(X, w, b)          # all predictions
    eps  = 1e-9                       # prevent log(0)
    J    = -np.mean(
               y * np.log(f + eps) +
               (1 - y) * np.log(1 - f + eps)
           )
    return J

J_before = compute_cost(X_n, y, w, b)
print(f"J before training = {J_before:.4f}")
print(f"  (0.693 = maximum uncertainty — model guesses 50% for everyone)")

print(f"\nHow J is computed for Maria (y=1, f=0.5):")
print(f"  y*log(f)       = 1 × log(0.5) = {1*np.log(0.5):.4f}")
print(f"  (1-y)*log(1-f) = 0 × log(0.5) = 0")
print(f"  contribution   = {-1*np.log(0.5):.4f}")


# ── STEP 6: GRADIENTS ────────────────────────────────────────
# dJ/dw = (1/m) × X_n.T · (f - y)  ← VECTOR of 3 gradients
# dJ/db = mean(f - y)               ← SCALAR
# Same formulas as your Austin house model.

print("\n\n🧮 STEP 6: Gradients — dJ/dw and dJ/db")
print("-" * 40)

def compute_gradient(X, y, w, b):
    m     = X.shape[0]
    f     = predict(X, w, b)          # predictions
    err   = f - y                     # errors

    dj_dw = np.dot(X.T, err) / m     # VECTOR — same formula as house model
    dj_db = np.mean(err)              # SCALAR — same formula

    return dj_dw, dj_db

dj_dw, dj_db = compute_gradient(X_n, y, w, b)

print(f"err = f - y:")
for i, name in enumerate(names):
    f_i = predict(X_n[i:i+1], w, b)[0]
    print(f"  {name:<6}: {f_i:.3f} - {y[i]} = {f_i-y[i]:+.3f}")

print(f"\ndj_dw = {dj_dw.round(3)}")
print(f"  dj_dw[0] = {dj_dw[0]:.3f}  ← budget gradient   → w₁ must go {'UP ✓' if dj_dw[0]<0 else 'DOWN'}")
print(f"  dj_dw[1] = {dj_dw[1]:.3f}  ← urgency gradient  → w₂ must go {'UP ✓' if dj_dw[1]<0 else 'DOWN'}")
print(f"  dj_dw[2] = {dj_dw[2]:.3f}  ← reply gradient    → w₃ must go {'DOWN ✓' if dj_dw[2]>0 else 'UP'}")
print(f"\ndj_db = {dj_db:.3f}")


# ── STEP 7: ONE UPDATE STEP ──────────────────────────────────
# w := w - alpha * dj_dw
# b := b - alpha * dj_db
# Same update rule as your Austin house model.

print("\n\n⬆️  STEP 7: One Gradient Descent Update")
print("-" * 40)

alpha = 0.5   # SCALAR — learning rate

print(f"alpha = {alpha}")
print(f"w before: {w}")
print(f"dj_dw:    {dj_dw.round(3)}")
print(f"\nw[0] = {w[0]} - {alpha} × ({dj_dw[0]:.3f}) = {w[0] - alpha*dj_dw[0]:.3f}")
print(f"w[1] = {w[1]} - {alpha} × ({dj_dw[1]:.3f}) = {w[1] - alpha*dj_dw[1]:.3f}")
print(f"w[2] = {w[2]} - {alpha} × ({dj_dw[2]:.3f}) = {w[2] - alpha*dj_dw[2]:.3f}")
print(f"b    = {b}   - {alpha} × ({dj_db:.3f}) = {b - alpha*dj_db:.3f}")

w = w - alpha * dj_dw   # update w (vector)
b = b - alpha * dj_db   # update b (scalar)

print(f"\nw after 1 step: {w.round(3)}")
print(f"b after 1 step: {b:.3f}")
print(f"J after 1 step: {compute_cost(X_n, y, w, b):.4f}  (was {J_before:.4f})")


# ── STEP 8: GRADIENT DESCENT LOOP ────────────────────────────
# Repeat steps 6 and 7 one thousand times.
# J falls. Model learns. Weights find the best values.
# Identical loop to your Austin house model.

print("\n\n🔄 STEP 8: Gradient Descent — 1000 steps")
print("-" * 40)

def gradient_descent(X, y, alpha, iters):
    w = np.zeros(X.shape[1])   # start at zero
    b = 0.0
    J_history = []              # track cost each step

    for i in range(iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw  # update all 3 weights at once
        b = b - alpha * dj_db  # update bias
        J_history.append(compute_cost(X, y, w, b))

    return w, b, J_history

w, b, J_history = gradient_descent(X_n, y, alpha=0.5, iters=1000)

print(f"J at start:      {0.6931:.4f}")
print(f"J after step 10: {J_history[9]:.4f}")
print(f"J after step 100:{J_history[99]:.4f}")
print(f"J after step 1000:{J_history[999]:.4f}")
print(f"\nLearned weights:")
print(f"  w = {w.round(3)}")
print(f"  w[0] = {w[0]:.3f}  ← budget   (positive = higher budget → sign ✓)")
print(f"  w[1] = {w[1]:.3f}  ← urgency  (positive = more urgent → sign ✓)")
print(f"  w[2] = {w[2]:.3f}  ← reply    (negative = slow reply → ghost ✓)")
print(f"  b    = {b:.3f}")


# ── STEP 9: DECISION BOUNDARY ────────────────────────────────
# Boundary is where z = 0  →  sigmoid = 0.5
# Formula: w[0]*x1 + w[1]*x2 + w[2]*x3 + b = 0
# Model predicts SIGN when z > 0
# Model predicts GHOST when z < 0

print("\n\n📏 STEP 9: Decision Boundary")
print("-" * 40)
print(f"Boundary equation: {w[0]:.2f}×budget + {w[1]:.2f}×urgency + {w[2]:.2f}×reply + {b:.2f} = 0")
print(f"\nLead is above boundary (predict SIGN) when z > 0")
print(f"Lead is below boundary (predict GHOST) when z < 0")

print(f"\nChecking all training leads against the boundary:")
preds_after = predict(X_n, w, b)
for i, name in enumerate(names):
    z_i = np.dot(X_n[i], w) + b
    prob = preds_after[i]
    pred = 1 if prob >= 0.5 else 0
    real = int(y[i])
    correct = "✓" if pred == real else "✗ WRONG"
    side = "ABOVE" if z_i > 0 else "BELOW"
    print(f"  {name:<6}: z={z_i:+.2f} ({side} boundary) → {prob:.0%} → pred={pred} real={real} {correct}")


# ── STEP 10: PREDICT NEW LEADS ───────────────────────────────
# Three new leads this week.
# Scale with SAME mu and sigma from training.
# Run through trained model. Get probability. Make decision.

print("\n\n🎯 STEP 10: Score New Leads")
print("-" * 40)

new_leads = [
    {"name": "Rachel", "features": [2.2, 4,  1.5]},
    {"name": "Carlos", "features": [1.3, 1,  30 ]},
    {"name": "Priya",  "features": [2.6, 5,  0.8]},
]

print(f"Using training mu    = {mu.round(3)}")
print(f"Using training sigma = {sigma.round(3)}")
print()

results = []
for lead in new_leads:
    raw      = np.array(lead["features"])  # raw features
    scaled   = (raw - mu) / sigma           # MUST use training mu/sigma
    z        = np.dot(scaled, w) + b        # linear step
    prob     = sigmoid(z)                   # probability
    decision = "✅ CALL FIRST"  if prob >= 0.8 else \
               "📞 Call today"  if prob >= 0.5 else \
               "⏳ Low priority"

    results.append((prob, lead["name"], decision))

    print(f"{lead['name']}:")
    print(f"  Raw features:    {raw}  (${raw[0]}k, urgency {int(raw[1])}, {raw[2]}h reply)")
    print(f"  Scaled features: {scaled.round(3)}")
    print(f"    budget:  ({raw[0]} - {mu[0]:.3f}) / {sigma[0]:.3f} = {scaled[0]:.3f}")
    print(f"    urgency: ({raw[1]} - {mu[1]:.3f}) / {sigma[1]:.3f} = {scaled[1]:.3f}")
    print(f"    reply:   ({raw[2]} - {mu[2]:.3f}) / {sigma[2]:.3f} = {scaled[2]:.3f}")
    print(f"  z = {w[0]:.2f}×{scaled[0]:.2f} + {w[1]:.2f}×{scaled[1]:.2f} + {w[2]:.2f}×{scaled[2]:.2f} + {b:.2f}")
    print(f"    = {w[0]*scaled[0]:.2f} + {w[1]*scaled[1]:.2f} + {w[2]*scaled[2]:.2f} + {b:.2f} = {z:.2f}")
    print(f"  sigmoid({z:.2f}) = {prob:.1%}")
    print(f"  Decision: {decision}")
    print()


# ── FINAL SUMMARY ────────────────────────────────────────────

print("=" * 55)
print("  CALL PRIORITY THIS WEEK")
print("=" * 55)
results.sort(reverse=True)
for rank, (prob, name, decision) in enumerate(results, 1):
    print(f"  {rank}. {name:<8} {prob:.0%} probability  {decision}")

print()
print("=" * 55)
print("  CONCEPTS USED IN THIS CODE")
print("=" * 55)
concepts = [
    ("Scalars",         "m, n, alpha, b, J — single numbers"),
    ("Vectors",         "w=[0,0,0], y=[1,0,1...], mu, sigma"),
    ("Matrix",          "X — shape (8,3), X_n — scaled matrix"),
    ("Indexing",        "X[0]=Maria, X[-1]=Dave, w[0]=budget w"),
    ("Slicing",         "X[:,0]=all budgets, X[:4]=first 4 leads"),
    ("Feature scaling", "(X - mu) / sigma — 3 lines"),
    ("Sigmoid",         "1/(1+e^-z) — squishes to 0-1"),
    ("np.dot",          "np.dot(X_n, w) + b — the prediction"),
    ("Cross-entropy J", "-mean(y*log(f)+(1-y)*log(1-f))"),
    ("dJ/dw",           "np.dot(X.T, err) / m — vector gradient"),
    ("dJ/db",           "np.mean(err) — scalar gradient"),
    ("Update rule",     "w = w - alpha * dj_dw"),
    ("Training loop",   "for i in range(1000): ... repeat"),
    ("Decision boundary","z=0 is where sigmoid=0.5"),
    ("Prediction",      "sigmoid(np.dot(scaled, w) + b)"),
]
for concept, description in concepts:
    print(f"  ✓ {concept:<20} {description}")

print()
print("  Every concept from every conversation — in one program.")

  AUSTIN APARTMENT LEAD SCORER
  Built from scratch using what we learned

📋 STEP 1: Training Data
----------------------------------------
X shape: (8, 3)  ← 8 leads, 3 features each
y shape: (8,)  ← 8 outcomes

First lead (indexing):  X[0] = [2.5 5.  0.5]  ← Maria
Last lead  (indexing):  X[-1] = [ 1.4  2.  20. ]  ← Dave
All budgets (slicing):  X[:,0] = [2.5 1.2 2.8 1.5 3.  1.1 2.3 1.4]
Signed leads (slicing): X[:1] = [[2.5 5.  0.5]]


📐 STEP 2: Feature Scaling (Z-score)
----------------------------------------
mu    = [ 1.975  3.    16.812]
         budget=1.975  urgency=3.000  reply=16.812

sigma = [ 0.71   1.581 17.062]
         budget=0.710  urgency=1.581  reply=17.062

How we got mu[0] = 1.975:
  2.5+1.2+2.8+1.5+3.0+1.1+2.3+1.4 = 15.799999999999999
  15.799999999999999 / 8 = 1.975

Maria raw:    [2.5 5.  0.5]
Maria scaled: [ 0.739  1.265 -0.956]
  budget:  (2.5 - 1.975) / 0.710 = 0.739
  urgency: (5.0 - 3.000) / 1.581 = 1.265
  reply:   (0.5 - 16.812) / 17.062 = -0.956

Full scal